# Setting up the environment

Create a virtual environment using:
```bash
conda create -n "yolo_training" python==3.8
```

and then activate it using:
```bash
conda activate yolo_training
```

# Installing the dependencies

Execute this in your terminal:
```bash
pip install tensorflow onnx-tf onnxruntime onnx onnxsim coremltools --no-deps
```

Clone the yolov7 repository:
```bash
git clone https://github.com/WongKinYiu/yolov7.git
```

Enter the repo and install the requirements:
```bash
cd yolov7
pip install -r requirements.txt
```

# Preparing the training data

1- Upload images and labels in the format of the tree :   
<pre>
  custom_dataset
  ├── images
  │   ├── train
  │   │   ├── train0.jpg
  │   │   └── train1.jpg
  │   ├── val
  │   │   ├── val0.jpg
  │   │   └── val1.jpg
  │   └── test (optional)
  │       ├── test0.jpg
  │       └── test1.jpg
  └── labels
      ├── train
      │   ├── train0.txt
      │   └── train1.txt
      ├── val
      │   ├── val0.txt
      │   └── val1.txt
      └── test (optional)
          ├── test0.txt
          └── test1.txt
</pre>


2- Create config.yaml
```yaml
train: ../custom_dataset/images/train # train images
test: ../custom_dataset/images/test # test images
val: ../custom_dataset/images/val # val images

# Classes
nc: 10 # number of classes
names: ['a', 'b', 'c', ..., 'z'] # class names
```

3- Edit number of classes in yolov7/cfg/training/yolov7-tiny.yaml
```yaml
# parameters
nc: 10  # number of classes (should be the same as in config.yaml)
depth_multiple: 1.0  # model depth multiple
width_multiple: 1.0  # layer channel multiple
......
```

4- The working directory should look like this now:
<pre>
  working_dir
  ├── custom_dataset
  │  ├── images
  │  │   ├── train
  │  │   ├── val
  │  │   └── test (optional)
  │  └── labels
  │      ├── train
  │      ├── val
  │      └── test (optional)
  ├── yolov7
  ├── config.yaml
  ├── prepare_data.ipynb
  └── training.ipynb
</pre>

# Training

Now that the data and config are ready we can train our model

Make sure to be inside the yolov7 folder and execute this command:
```bash
python train.py --img-size 640 --cfg cfg/training/yolov7-tiny.yaml --hyp data/hyp.scratch.custom.yaml --batch 8 --epochs 100 --data ../config.yaml --weights yolov7-tiny.pt --name yolo_det_model
```

# Resume training (if interrupted)

If the training was interrupted (intentionnally or not) you can resume where it stopped by executing this:
```bash
python train.py --img 640 --batch 8 --epochs 100 --resume runs/train/yolo_det_model/weights/last.pt
```

# Evaluate the model

In order to evaluate your model manually (It's automatically evaluated if the training is not interrupted) execute this command:
```bash
python test.py --img 640 --batch 8 --data ../config.yaml --weights runs/train/yolo_det_model/weights/best.pt --task test --device cpu
```

# Test your model on a single image

Let's see if the model you trained is good, execute this:
```bash
python detect.py --weights runs/train/yolo_det_model/weights/best.pt --source ../whatever-your-image-name-is.jpg  --device cpu
```

**NB: Make sure the image is outside the yolov7 repository**

# Convert the model to .onnx format

In order to get a TFLite file we need to convert our model to onnx in a first time:

```bash
python export.py --weights runs/train/yolo_det_model/weights/best.pt --grid --end2end --iou-thres 0.60 --topk 1 --conf-thres 0.60 --img-size 640 640 --max-wh 640
```

**NB:**
```
--topk is the number of best detections to take (takes the n best detections)
--iou-thresh is the Intersection over Union threshold
--conf-thresh is the confidence threshold (all detections that have a confidence below that threshold are not taken into consideration)  
```

# Testing the .onnx model

Here we define a helper function called **letterbox**, this function pads or resizes the image to 640x640 which is the maximum size the YOLO v7 model can accept. Apart from these we also define the class names and colors for visualization purposes.

In [ ]:
import random, cv2
import numpy as np

model_path = "yolov7/runs/train/yolo_det_model/weights/best.onnx"
image_path = "your-test-image-path"

def letterbox(im, new_shape=(640, 640), color=(114, 114, 114), auto=True, scaleup=True, stride=32):
    # Resize and pad image while meeting stride-multiple constraints
    shape = im.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:  # only scale down, do not scale up (for better val mAP)
        r = min(r, 1.0)

    # Compute padding
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding

    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, stride), np.mod(dh, stride)  # wh padding

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return im, r, (dw, dh)

#Name of the classes according to class indices.
names = ["card"]
#Creating random colors for bounding box visualization.
colors = {name:[random.randint(0, 255) for _ in range(3)] for i,name in enumerate(names)}

In [ ]:
import onnxruntime as ort
from matplotlib import pyplot as plt

cuda = False

#Loading the ONNX inference session.
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if cuda else ['CPUExecutionProvider']
session = ort.InferenceSession(model_path, providers=providers)

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

#Preprocessing the image for prediction.
image = img.copy()
image, ratio, dwdh = letterbox(image, auto=False)
image = image.transpose((2, 0, 1))
image = np.expand_dims(image, 0)
image = np.ascontiguousarray(image)

im = image.astype(np.float32)
im /= 255
im.shape

#Getting onnx graph input and output names.
outname = [i.name for i in session.get_outputs()]
inname = [i.name for i in session.get_inputs()]
inp = {inname[0]:im}

# Running inference using session.
outputs = session.run(outname, inp)[0]


ori_images = [img.copy()]

#Visualizing bounding box prediction.
for i,(batch_id,x0,y0,x1,y1,cls_id,score) in enumerate(outputs):
    image = ori_images[int(batch_id)]
    box = np.array([x0,y0,x1,y1])
    box -= np.array(dwdh*2)
    box /= ratio
    box = box.round().astype(np.int32).tolist()
    cls_id = int(cls_id)
    score = round(float(score),3)
    name = names[cls_id]
    color = colors[name]
    name += ' '+str(score)
    cv2.rectangle(image,box[:2],box[2:],color,2)
    cv2.putText(image,name,(box[0], box[1] - 2),cv2.FONT_HERSHEY_SIMPLEX,0.75,[225, 255, 255],thickness=2)  

plt.imshow(ori_images[0])

# Converting the .onnx model to a TensorFlow model (pb)

In order to do so, we need to execute this command:
```bash
onnx-tf convert -i runs/train/yolo_det_model/weights/best.onnx -o ../tf_yolo_det
```

# Now we convert the TensorFlow model to its TFLite version

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('tf_yolo_det')
tflite_model = converter.convert()

with open('yolo_detector.tflite', 'wb') as f:
  f.write(tflite_model)

# Finally, let's test our TFLite model

In [ ]:
# Load the TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path="yolo_detector.tflite")
import time

#Allocate tensors.
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = input_details[0]['shape']
cumul = []
# Test the model
# folder = 'dev'
# for image in os.listdir(folder):
start_time = time.time()
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
image, ratio1, dwdh = letterbox(img, auto=False)
image = image.transpose((2, 0, 1))
image = np.expand_dims(image, 0)
# image = np.ascontiguousarray(image)
im = image.astype(np.float32)
im /= 255

interpreter.set_tensor(input_details[0]['index'], im)
interpreter.invoke()
# The function `get_tensor()` returns a copy of the tensor data.
# Use `tensor()` in order to get a pointer to the tensor.
output_data = interpreter.get_tensor(output_details[0]['index'])
output_data.shape
elapsed = time.time()- start_time
# cumul.append(time.time()- start_time)
elapsed

In [ ]:
# Vusialization
for i,(batch_id,x0,y0,x1,y1,cls_id,score) in enumerate(output_data):
    box = np.array([x0,y0,x1,y1])
    box -= np.array(dwdh*2)
    box /= ratio1
    box = box.round().astype(np.int32).tolist()
    cls_id = int(cls_id)
    score = round(float(score),3)
    name = names[cls_id]
    color = colors[name]
    name += ' '+str(score)
    cv2.rectangle(img,box[:2],box[2:],color,2)
    cv2.putText(img,name,(box[0], box[1] - 2),cv2.FONT_HERSHEY_SIMPLEX,0.75,[0, 0, 0],thickness=2)  
plt.imshow(img)